# 04. Construção da Base de Conhecimento Vetorial (ChromaDB & Embeddings)

**Contexto:** Tech Challenge 04 - Voice of Customer Intelligence com RAG  
**Requisitos do Desafio:** Criação de Embeddings, Armazenamento, Indexação e Exploração Contextual com Metadados.  

Nesta etapa, conectamos o corpus textual enriquecido e pré-processado (`reviews_enriched_for_rag.csv`) a um **Banco de Dados Vetorial (ChromaDB)**. Cada avaliação e seus metadados de compra/categoria são convertidos em representações vetoriais densas (*Embeddings*) e indexados com métrica de similaridade por **cosseno**.

In [ ]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
import os
from tqdm import tqdm

DATA_DIR = '../data/'
PROCESSED_DIR = '../data/processed/'
DB_PATH = '../data/vector_db/'

# Carregamento do dataset pré-processado e consolidado
df_docs = pd.read_csv(f'{PROCESSED_DIR}reviews_enriched_for_rag.csv')
print(f"Total de documentos para indexação: {len(df_docs):,}")

## 1. Configuração do ChromaDB e Modelo de Embeddings

**Decisão Arquitetural e de Custo:**  
Para indexar mais de 40.000 avaliações sem incorrer em custos proibitivos de API externa ou limites de taxa (rate limits), utilizamos a função `DefaultEmbeddingFunction()` do ChromaDB (`all-MiniLM-L6-v2` via `sentence-transformers`).  
- Execução 100% local e eficiente em CPU/GPU.
- Espaço métrico configurado com **similaridade de cosseno** (`hnsw:space: cosine`).

In [ ]:
# Inicialização do cliente persistente do ChromaDB
client = chromadb.PersistentClient(path=DB_PATH)
sentence_transformer_ef = embedding_functions.DefaultEmbeddingFunction()

collection_name = "olist_reviews"
collection = client.get_or_create_collection(
    name=collection_name,
    embedding_function=sentence_transformer_ef,
    metadata={"hnsw:space": "cosine"}
)

print(f"Coleção '{collection_name}' pronta. Registros atuais: {collection.count():,}")

## 2. Indexação dos Documentos em Lotes (Batches)
Para garantir eficiência de memória e tolerância a falhas, a ingestão no ChromaDB é realizada em lotes de 1.000 documentos, armazenando os metadados (`review_score`) associados a cada vetor.

In [ ]:
batch_size = 1000
total_docs = len(df_docs)

if collection.count() < total_docs:
    print("Iniciando indexação dos documentos no ChromaDB...")
    for i in tqdm(range(0, total_docs, batch_size)):
        batch = df_docs.iloc[i:i+batch_size]
        
        documents = batch['rag_document'].tolist()
        ids = batch['review_id'].tolist()
        metadatas = [{"review_score": int(score)} for score in batch['review_score'].tolist()]
        
        collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
    print(f"\nIndexação concluída! Total indexado: {collection.count():,} registros.")
else:
    print(f"A base já contém {collection.count():,} registros indexados. Etapa de inserção concluída.")

## 3. Teste de Recuperação com Filtragem de Metadados (Desafio Adicional)
Validamos a recuperação vetorial aplicando filtros contextuais sobre a nota (`where={"review_score": {"$lte": 2}}`) para eliminar falsos positivos semânticos em queixas.

In [ ]:
query_teste = "Atraso na entrega e suporte ruim"
resultados = collection.query(
    query_texts=[query_teste],
    n_results=3,
    where={"review_score": {"$lte": 2}}
)

print(f"Consulta realizada: '{query_teste}'")
print("\n=== Evidências Recuperadas (Filtro review_score <= 2) ===")
for i, (doc, doc_id, meta) in enumerate(zip(resultados['documents'][0], resultados['ids'][0], resultados['metadatas'][0]), 1):
    print(f"[{i}] ID: {doc_id} | Metadados: {meta}")
    print(f"    {doc}")
    print("-"*60)